# 09 — Controllers

Side effects on data changes with `data_controller`.

**What you learn:**
- `data_controller`: runs a function for side effects (no data output)
- Unlike `data_formula`, a controller does not write to a data path
- Controllers fire on every change of their dependencies
- Use cases: logging, validation, notifications, syncing

**Note:** `_delay` (debounce) and `_interval` (periodic) require an async event loop. See example 11.

**Prerequisites:** 08_reactive_basics

In [ ]:
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager

log = []


class LoggingApp(ReactiveManager):
    def __init__(self):
        self.page = self.set_builder('page', HtmlBuilder)
        self.run(subscribe=True)

    def store(self, data):
        data['counter'] = 0
        data['name'] = 'Alice'

    def main(self, source):
        source.data_controller(
            func=lambda counter: log.append(f'counter={counter}'),
            counter='^counter',
        )
        source.data_controller(
            func=lambda name: log.append(f'name={name}'),
            name='^name',
        )
        source.body().p('^counter')

app = LoggingApp()
print(f'After build: {log}')

In [ ]:
log.clear()
app.reactive_store['counter'] = 1
app.reactive_store['counter'] = 2
app.reactive_store['counter'] = 3
print(f'After 3 counter changes: {log}')

In [ ]:
log.clear()
app.reactive_store['name'] = 'Bob'
print(f'After name change: {log}')

## Controller with multiple dependencies

A controller watching two values fires when either changes:

In [ ]:
alerts = []

class AlertApp(ReactiveManager):
    def __init__(self):
        self.page = self.set_builder('page', HtmlBuilder)
        self.run(subscribe=True)

    def store(self, data):
        data['temperature'] = 20
        data['threshold'] = 30

    def main(self, source):
        source.data_controller(
            func=lambda temperature, threshold: alerts.append(
                f'ALERT: {temperature}C' if temperature > threshold else f'OK: {temperature}C'
            ),
            temperature='^temperature', threshold='^threshold',
        )
        source.body().p('^temperature')

app2 = AlertApp()
alerts.clear()
app2.reactive_store['temperature'] = 25
app2.reactive_store['temperature'] = 35  # exceeds threshold
app2.reactive_store['threshold'] = 40    # raise threshold
print(f'Alerts: {alerts}')